<a href="https://colab.research.google.com/github/JosuBarrado/03MIAR_Algortimos-de-Optimizacion---2025/blob/main/Trabajo_Practico_Josu_Barrado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Josu Barrado Guezala  <br>
Url: https://github.com/JosuBarrado/03MIAR_Algortimos-de-Optimizacion---2025.git
Google Colab: https://colab.research.google.com/drive/1S1-6Np1cvGM8CyviH7bXgTWP3fMG3SAm?usp=sharing    
Problema:

>1. Sesiones de doblaje    

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en
las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el
estudio de grabación independientemente del número de tomas que se graben. No es
posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de
manera que el gasto por los servicios de los actores de doblaje sea el menor posible.     
Los datos son:    
Número de actores: 10
Número de tomas : 30
Actores/Tomas
:
https://bit.ly/36D8IuK
- 1 indica que el actor participa en la toma
- 0 en caso contrario








                                        

In [2]:
import pandas as pd
import numpy as np
import random
import os

#Modelo
- ¿Cómo represento el espacio de soluciones?    

El espacio de soluciones será un lista anidada, con la solución de tomas a grabar, cada anidación tiene 6 elementos y corresponde a un día. Esta solución tiene que venir calculada a partir de le búsqueda de un coste mínimo de desplazamiento de actores. Esta lista tomará valores únicos (no repetidos) del 1 al 30. Además, se creará otra lista que tendrá un set para cada día, donde se unen los actores que participan en las tomas seleccionadas para eses día (union de conjuntos). El uso de set se realiza de forma intencionada, porque no permite repeticiones y permite funciones como unión de conjuntos, diferencia... Llamaré a esta representación de la solución plan    
$$plan =
\begin{bmatrix}
t_{11} & ··· & t_{16} \\
··· & ··· & ··· \\
t_{51} & ··· & t_{56}
\end{bmatrix}
$$

- ¿Cual es la función objetivo?    

La función objetivo del problema, que traduce el espacio de soluciones (plan) a coste real, será la que cuente para cada día cuantos actores diferentes han tenido que participar. Suponiendo que el resultado se almacenará como un plan
Representado de forma matemática queda:    
$$f(\text{plan}) = \sum_{d=1}^{5} \left| \bigcup_{t \in D_d} A_t \right|$$

- ¿Como implemento las restricciones?    

Las restricciones son que los actores deben coincidir en las tomas que así lo indican y no se pueden hacer más de seis tomas por día.    
La restricción de seis tomas vendrá dada por el modo de representar el espacio de soluciones.    
La restricción de coincidencia de actores vendrá dada por la creación de un diccionario a partir de la tabla de datos de entrada del problema que tendrá la estructura:

datos_dict = {toma : {act1, act2,...}} → Cada key del diccionario será una toma y tendrá como value un set con los actores que participan en dicha toma. Al generar las posibles soluciones solo se podrá seleccionar una toma de esta diccionario que automáticamente incluirá los actores necesarios si o si.

In [3]:
# Aunque esta parte de código la he realizado en local, como seguramente haya problemas para compartir el archivo desde Colab,
# Copia el diccionario que he conseguido directamente para poder trabajar con el. Es exactamente la tabla del problema
dict_actores = {1: {'act1', 'act2', 'act3', 'act4', 'act5'},
 2: {'act3', 'act4', 'act5'},
 3: {'act2', 'act5', 'act7'},
 4: {'act1', 'act2', 'act7', 'act8'},
 5: {'act2', 'act4', 'act8'},
 6: {'act1', 'act2', 'act4', 'act5'},
 7: {'act1', 'act2', 'act4', 'act5'},
 8: {'act1', 'act2', 'act6'},
 9: {'act1', 'act2', 'act4'},
 10: {'act1', 'act2', 'act6', 'act9'},
 11: {'act1', 'act2', 'act3', 'act5', 'act8'},
 12: {'act1', 'act2', 'act3', 'act4', 'act6'},
 13: {'act1', 'act4', 'act5'},
 14: {'act1', 'act3', 'act6'},
 15: {'act1', 'act2', 'act7'},
 16: {'act10', 'act4'},
 17: {'act1', 'act3'},
 18: {'act3', 'act6'},
 19: {'act1', 'act3'},
 20: {'act1', 'act3', 'act4', 'act5'},
 21: {'act6', 'act8'},
 22: {'act1', 'act2', 'act3', 'act4'},
 23: {'act1', 'act3'},
 24: {'act3', 'act6'},
 25: {'act1', 'act10', 'act2', 'act4'},
 26: {'act1', 'act3', 'act5', 'act9'},
 27: {'act4', 'act5'},
 28: {'act1', 'act4'},
 29: {'act1', 'act5', 'act6'},
 30: {'act1', 'act4'}}
display(dict_actores)


{1: {'act1', 'act2', 'act3', 'act4', 'act5'},
 2: {'act3', 'act4', 'act5'},
 3: {'act2', 'act5', 'act7'},
 4: {'act1', 'act2', 'act7', 'act8'},
 5: {'act2', 'act4', 'act8'},
 6: {'act1', 'act2', 'act4', 'act5'},
 7: {'act1', 'act2', 'act4', 'act5'},
 8: {'act1', 'act2', 'act6'},
 9: {'act1', 'act2', 'act4'},
 10: {'act1', 'act2', 'act6', 'act9'},
 11: {'act1', 'act2', 'act3', 'act5', 'act8'},
 12: {'act1', 'act2', 'act3', 'act4', 'act6'},
 13: {'act1', 'act4', 'act5'},
 14: {'act1', 'act3', 'act6'},
 15: {'act1', 'act2', 'act7'},
 16: {'act10', 'act4'},
 17: {'act1', 'act3'},
 18: {'act3', 'act6'},
 19: {'act1', 'act3'},
 20: {'act1', 'act3', 'act4', 'act5'},
 21: {'act6', 'act8'},
 22: {'act1', 'act2', 'act3', 'act4'},
 23: {'act1', 'act3'},
 24: {'act3', 'act6'},
 25: {'act1', 'act10', 'act2', 'act4'},
 26: {'act1', 'act3', 'act5', 'act9'},
 27: {'act4', 'act5'},
 28: {'act1', 'act4'},
 29: {'act1', 'act5', 'act6'},
 30: {'act1', 'act4'}}

In [4]:
# Espacio de soluciones prueba aleatoria
plan = np.random.choice(range(1, 31), 30).reshape(5, 6)
display(plan)

array([[17, 11, 30, 19, 19, 11],
       [ 3, 12,  6, 15,  9,  6],
       [27, 16, 17,  4,  1, 23],
       [ 1,  5, 12,  7, 30,  3],
       [21, 24,  7, 14,  3, 16]])

In [7]:
# función objetivo → cálculo de coste por día y plan completo
def coste_dia(tomas_dia, dict_actores):
    actores = set()
    for t in tomas_dia:
        actores |= dict_actores[t]
    return len(actores)

def coste_total(plan, dict_actores):
    coste_total = 0
    for dia in plan:
        coste_total += coste_dia(dia, dict_actores)
    return coste_total

# Pruebo mi solución aleatoria con el cálculo del coste
coste_total(plan, dict_actores)

38

In [8]:
# planteo un greedy inicial

def greedy_inicial(dict_actores, n_dias=5, tomas_por_dia=6):
    restantes = {k: set(v) for k, v in dict_actores.items()}
    plan = []

    # recorro cada día
    for j in range(n_dias):
        # semilla: toma con más actores al comienzo de cada día
        max_key = max(restantes, key=lambda t: len(restantes[t]))
        dia = [max_key]
        # lo almaceno y elimino selección con más actores
        actores_dia = set(restantes.pop(max_key))

        # hasta llenar el máximo de tomas por día
        while len(dia) < tomas_por_dia:
            mejor_toma = None
            mejor_delta = np.inf

            # recorro  cada elemento del diccionario restante para
            # comprobar el siguiente mejor paso
            for t, acts in restantes.items():
                # union del conjunto y resta con el actual para tener
                # el delta de actores con esta selección
                delta = len((actores_dia | acts) - actores_dia)

                # minimizo actores añadidos seleccionando
                # el menor delta
                if delta < mejor_delta:
                    mejor_delta = delta
                    mejor_toma = t
            # Una vez seleccionado el mejor, lo añado a la solución
                # array que toma las tomas seleccionadas para el dia
            dia.append(mejor_toma)
                # set de actores sacando la mejor toma desde restantes
            actores_dia |= restantes.pop(mejor_toma)

        plan.append(dia)

    return plan

plan_greedy = greedy_inicial(dict_actores)
coste_greedy = coste_total(plan_greedy, dict_actores)
print(f"El plan voraz inicial queda:")
display(plan_greedy)
print(f"Coste greedy inicial: {coste_greedy}")


El plan voraz inicial queda:


[[1, 2, 6, 7, 9, 13],
 [11, 17, 19, 23, 3, 4],
 [12, 8, 14, 18, 22, 24],
 [10, 15, 21, 5, 28, 30],
 [20, 27, 16, 25, 26, 29]]

Coste greedy inicial: 31


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones    

### Espacio de soluciones:   
Las soluciones se calculan mediante combinatoria. Se trata de un problema de combinación sin repetición, realizado para 5 grupos.     
$$\binom{n}{k}=\frac{n!}{k!(n-k)!}$$

En el primer grupo tendremos, 30 elementos y tomamos 6 → del total de 30 tomas escogemos 6 posibles para un día. n = 30, k = 6.    
$$\binom{30}{6}=\frac{30!}{6!(30-6)!}$$

Para la siguiente selección, ya se habrán escogido 6, por lo que quedan 24 elemento y se vuelven a escoger 6.    
$$\binom{24}{6}=\frac{24!}{6!(24-6)!}$$
Y así con los cinco elementos, lo que queda en:    
$$\binom{30}{6}\binom{24}{6}\binom{18}{6}\binom{12}{6}\binom{6}{6}$$
$$\frac{30!}{6!\,24!}\cdot\frac{24!}{6!\,18!}\cdot\frac{18!}{6!\,12!}\cdot\frac{12!}{6!\,6!}\cdot\frac{6!}{6!\,0!}$$

Pero los grupos que se han escogido en cada iteración pueden intercambiarse y la solución será la misma, por lo que hay que dividir entre $5!$ para omitir estos casos. Simplificando términos queda:
$$\frac{30!}{(6!)^5\,5!}=1.14\times10^{16}$$

### Orden de complejidad:    
La resolución de este problema, si se hace una búsqueda exhaustiva de todas sus posibles soluciones, es un problema **NP-Hard**. No se espera por tanto disponer de un algoritmo que encuentre la solución exacta en tiempo polinómico, lo que va a definir la decisión posterior de que algoritmo utilizar, de forma justificada.


#Diseño
- ¿Que técnica utilizo? ¿Por qué?    
Para este problema no es viable utilizar una técnica exacta para encontrar la solución óptima debido a la complejidad **NP-Hard** del problema, por lo que hay que utilizar alguna técnica heurística que permita llegar a una buena solución sin necesitar tiempos de ejecución algorítmicos elevados. He estudiado las diferentes técnicas posibles y consultado con la inteligencia artificial, y me ha parecido la más adecuada la de recocido simulado, ya que debido a mis estudios en mecatrónica industrial y posterior ingeniería eléctrica, me han llevado a conocer y estudiar esta técnica. Por lo tanto me parece muy interesante probar esta técnica de forma simulada para un problema de optimización.    
Pero para empezar desde un punto con esta técnica, voy a utilizar la solución obtenida con greedy.    

Esta técnica algorítmica requiere alta probabilidad (temperatura alta) de elección de peores soluciones al comienzo y a medida que avanza la búsqueda de soluciones, baja probabilidad (temperatura baja → cristalización con propiedades específicas).    

Para este caso concreto, como ya parto de una solución inicial, lo que propongo es intercambiar tomas entre diferentes días y elegir intercambios desfavorables al comienzo (con idea de escapar de mínimos locales) y a medida que se vaya avanzando quedarse solo con los mejores cambios posibles.

In [9]:
# Función para swap de tomas de forma aleatoria
def swap_aleatorio(plan):
    # comprehension list del plan para trabajar con copia
    nuevo = [dia[:] for dia in plan]
    # genero dios números aleatorios para seleccionar el día a cambiar
    d1, d2 = random.sample(range(len(nuevo)), 2)
    # Y dentro de cada día seleccionado aleatoriamente,
    # seleccionar una toma aleatoriamente también
    td1 = random.randrange(len(nuevo[d1]))
    td2 = random.randrange(len(nuevo[d2]))

    # Y hago el swap
    nuevo[d1][td1], nuevo[d2][td2] = nuevo[d2][td2], nuevo[d1][td1]

    return nuevo, d1, d2

Ahora hay que empezar por una temperatura inicial alta, y para que el valor tenga sentido en el rango numérico del problema, existe una fórmula que permite calcular su valor.    
$$T_0=-\frac{\overline{empeoramiento}}{\ln(p_0)}$$
En este fórmula debemos elegir un valor p0 que será la probabilidad inicial deseada, pongamos un 0.9. Y $\overline{empeoramiento}$ es la media de los empeoramientos positivos, para obtener esta media hay que hacer varios swaps mediante la función anterior y calcular su empeoramiento y hacer la media.

In [10]:
# Función para estimar la temperatura inicial
def t_init(plan, dict_actores, muestras= 400, p0= 0.9):
    # obtengo el coste inicial
    coste0= coste_total(plan, dict_actores)
    #inicializo empeoramientos positivos
    delta_pos= []

    # bucle sobre muestras elegidas
    for _ in range(muestras):
        vecino, _, _ = swap_aleatorio(plan)
        delta = coste_total(vecino, dict_actores) - coste0
        if delta > 0:
            delta_pos.append(delta)
    # Para asegurar devolver algo en caso de que no haya habido empeoramiento
    if not delta_pos:
        return 1.0

    media_delta = np.mean(delta_pos)
    T0 = -media_delta/np.log10(p0)
    return T0
t_init(plan, dict_actores)

np.float64(25.827862658925174)

Con la función para tener la temperatura inicial, la función de hacer el swap, podemos pasar a la función principal de recocido simulado. Aquí hay que utilizar unas funciones matemáticas importantes, la primera para calcular la probabilidad entre 0 y 1, en base a T
$$e^{(-\delta/T)}$$
donde $\delta$ es el el cambio que hay hacia la siguiente solución y T tomará valores cambiantes, voy a utilizar el criterio exponencial, realizando este cálculo en cada iteración
$$T_{k+1}=\alpha\,T_k$$
Donde alpha lo elijo como 0.995 y generará un decrecimiento exponencial de T

In [11]:
def recocido_simulado(dict_actores, plan_inicial, alpha= 0.995, Level= 500, Tmin= 1e-3, max_it= 800, seed= 42):
    # aseguro la semilla aleatoria para reproducibilidad
    random.seed(seed)
    # Inicializo variable actual como copia
    actual = [dia[:] for dia in plan_inicial]
    coste_actual = coste_total(actual, dict_actores)
    # inicializar variable mejor como copia
    mejor = [dia[:] for dia in actual]
    coste_mejor = coste_actual
    # Estimar la temperatura inicial mediante la función creada
    T = t_init(actual, dict_actores)

    # variable contador sin mejorar inicializada
    cnt_no_mejora = 0

    while T > Tmin and cnt_no_mejora < max_it:
        # cada bucle le digo que no mejora y si tras la comprobación mejora, cambia su valor
        # para gestionar el contador de iteraciones sin mejorar
        mejora = False

        # En cada etapa con temperatura, se realizan Level iteraciones
        # Permite generar más caos inicial y no realizar solo unos pocos swap
        for _ in range(Level):
            vecino, d1, d2 = swap_aleatorio(actual)

            # Para cálculos más eficientes, como solo se intercambian 2 tomas
            # basta con calcular la diferencia que generan en el día que se cambia
            coste_act_parcial = (coste_dia(actual[d1], dict_actores)
                                 + coste_dia(actual[d2], dict_actores))
            coste_vecino_parcial = (coste_dia(vecino[d1], dict_actores)
                                 + coste_dia(vecino[d2], dict_actores))
            coste_vecino = coste_actual - coste_act_parcial + coste_vecino_parcial
            delta = coste_vecino - coste_actual

            # Si mejora se acepta directamente
            if delta <= 0:
                actual = vecino
                coste_actual = coste_vecino

                if coste_actual < coste_mejor:
                    mejor = [dia[:] for dia in actual]
                    coste_mejor = coste_actual
                    mejora = True

            # Si empeora, hago uso de la probabilidad con temperatura
            else:
                prob = np.exp(-delta/T)
                # comparo con float [0, 1] aleatorio
                # prob alta al comienzo
                if random.random() < prob:
                    actual = vecino
                    coste_actual = coste_vecino
        if mejora:
            cnt_no_mejora = 0
        else:
            cnt_no_mejora += 1
        # cambia de alpha por método exponencial
        T *= alpha

    return mejor, coste_mejor

Con las funciones necesarias creadas, falta realizar su llamada y obtener un resultado para ver si mejora

In [12]:
plan_greedy = greedy_inicial(dict_actores)
coste_greedy = coste_total(plan_greedy, dict_actores)
print("Coste greedy inicial:", coste_greedy)

plan_mejor, coste_mejor = recocido_simulado(dict_actores, plan_greedy)
print("El plan de tomas por día óptimo obtenido es:")
display(plan_mejor)
print(f"Con un coste asociado de : {coste_mejor}")

Coste greedy inicial: 31
El plan de tomas por día óptimo obtenido es:


[[19, 24, 14, 18, 23, 17],
 [16, 27, 7, 30, 25, 13],
 [22, 2, 1, 9, 12, 20],
 [21, 10, 11, 29, 26, 8],
 [4, 28, 15, 6, 3, 5]]

Con un coste asociado de : 27
